# Lab 7: MNIST CNN in PyTorch (GPU)
AI Lab, University of Technology Bahrain. About 15 minutes. GPU.

**You will learn**
- The same CNN as lab 3, written in PyTorch
- Moving tensors and models to the GPU with `.to(device)`
- Writing the training loop yourself

**Before you run**
- On Kaggle: right-hand panel, *Settings → Accelerator → GPU T4 x2* (when the lab needs a GPU) and *Settings → Internet → On*. GPU and Internet need a phone-verified account.
- On Colab: *Runtime → Change runtime type → T4 GPU*.
- Run the cells from top to bottom with **Shift + Enter**.

In [ ]:
# Setup: run this cell first
import os, sys, time, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU: see 'Before you run' above")
torch.manual_seed(42)
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content" if "google.colab" in sys.modules else "."
DATA = os.path.join(BASE, "data")

In [ ]:
tf_ = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
try:
    train_ds = datasets.MNIST(DATA, train=True, download=True, transform=tf_)
    test_ds = datasets.MNIST(DATA, train=False, download=True, transform=tf_)
except Exception as err:
    raise RuntimeError("MNIST did not download. On Kaggle switch on Settings > Internet, then run this cell again.") from err
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=1000, num_workers=2)
print(len(train_ds), "train /", len(test_ds), "test")

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(64 * 5 * 5, 10)

    def forward(self, x):                              # x: (B, 1, 28, 28)
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)     # (B, 32, 13, 13)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)     # (B, 64, 5, 5)
        return self.fc(self.drop(x.flatten(1)))        # (B, 10) logits

model = Net().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model, "\nparameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
def evaluate():
    model.eval(); correct = 0
    with torch.no_grad():
        for x, y in test_dl:
            correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
    return correct / len(test_ds)

losses = []
for epoch in range(1, 6):
    model.train(); t = time.time(); running = 0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
        running += loss.item()
    losses.append(running / len(train_dl))
    print(f"epoch {epoch}  loss {losses[-1]:.4f}  test acc {evaluate():.4f}  ({time.time()-t:.1f}s)")

plt.plot(range(1, 6), losses, marker="o"); plt.xlabel("epoch"); plt.ylabel("train loss"); plt.show()
torch.save(model.state_dict(), os.path.join(BASE, "mnist_cnn.pt"))

## Try it
1. Run once with the GPU and once on CPU. Note the seconds per epoch.
2. Kaggle's *GPU T4 x2* gives you two GPUs (Colab gives one). Wrap the model in `nn.DataParallel(model)`. Does it get faster for such a small model?
3. Add `torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)` and call `scheduler.step()` after each epoch.

### Save your work
- **Kaggle:** *Save Version → Quick Save* keeps your edits; *Save & Run All* also keeps the outputs. Files written to `/kaggle/working` appear under **Output**.
- **Colab:** *File → Save a copy in Drive*. Files in `/content` are deleted when the session ends, so download anything you need from the Files panel first.